# GNNによる不正検知入門 (Introduction to GNN-based Fraud Detection)

## 導入

金融取引における不正検知は、マネーロンダリング対策、クレジットカード詐欺防止、仮想通貨の不正利用検出など、現代の金融システムにおける重要な課題です。従来の機械学習手法は個々の取引の特徴量（金額、時刻、頻度など）に基づいて不正を判定してきましたが、これらの手法は取引間の **関係性（グラフ構造）** を十分に活用できていませんでした。

**グラフニューラルネットワーク（GNN）** は、ノード（取引やアカウント）の特徴量とエッジ（取引関係）の両方を同時に学習できるディープラーニングの一分野であり、不正検知において以下の利点があります:

- **近傍情報の集約**: 不正ノードは他の不正ノードと接続していることが多く、GNNはこのパターンを捕捉可能
- **メッセージパッシング**: 近傍ノードからの情報を集約することで、個々の特徴だけでは見えない不正パターンを検出
- **半教師あり学習**: ラベル付きデータが少ない状況でも、グラフ構造を通じたラベル伝搬が可能

### Elliptic Bitcoin Dataset

本ノートブックでは、代表的な金融不正検知データセットである **Elliptic Bitcoin Dataset**（Weber et al., 2019）をモデルに、GNNによる不正検知の基本的なワークフローを学びます。このデータセットはビットコインのトランザクショングラフであり、各ノード（取引）が「不正（illicit）」「合法（licit）」「不明（unknown）」のいずれかにラベル付けされています。

実際の実装では、教育目的で合成データを使用しますが、データの構造とモデルの設計はEllipticデータセットの特性を反映しています。

### 関連ドキュメント

- [03-computer-science.md](../03-computer-science.md) — 計算機科学におけるナレッジグラフとネットワーク科学の基礎（GNN、グラフ埋め込みを含む）
- [05-emerging-fields.md](../05-emerging-fields.md) — 新興分野における不正検知・AMLへのGNNの適用

## 環境セットアップ

必要なライブラリをインストールします。既にインストール済みの場合はスキップされます。

In [ ]:
# 必要なパッケージのインストール
!pip install torch torch-geometric scikit-learn pandas matplotlib networkx

## ライブラリのインポート

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, GATConv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)
from sklearn.model_selection import train_test_split

# 乱数シードの固定（再現性のため）
torch.manual_seed(42)
np.random.seed(42)

# 日本語フォントの設定（環境に応じて変更してください）
# macOS の場合
plt.rcParams['font.family'] = 'Hiragino Sans'
# Windows の場合は以下を使用:
# plt.rcParams['font.family'] = 'MS Gothic'
# Linux の場合は以下を使用:
# plt.rcParams['font.family'] = 'IPAGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

# GPUが利用可能な場合はGPUを使用
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用デバイス: {device}")
print(f"PyTorch バージョン: {torch.__version__}")
print("ライブラリのインポートが完了しました。")

## Elliptic Bitcoin Dataset の概要

Elliptic Bitcoin Dataset は、ビットコインブロックチェーンから抽出された実世界のトランザクションネットワークであり、以下の特徴を持ちます:

| 項目 | 値 |
|---|---|
| ノード数（トランザクション） | **203,769** |
| エッジ数（支払いフロー） | **234,355** |
| 特徴量の次元数 | **166**（ローカル特徴 94 + 集約特徴 72） |
| ラベル | **illicit（不正）** / **licit（合法）** / **unknown（不明）** |
| タイムステップ | 49 時点（約2週間分） |

### 特徴量の構成

- **ローカル特徴（94次元）**: タイムステップ、入出力数、取引手数料、出力ボリューム、集約統計量など
- **集約特徴（72次元）**: 1ホップ先の近傍ノードから集約された同様の特徴量

### ラベル分布

- **不正（illicit）**: 4,545 ノード（約2%） — ランサムウェア、マルウェア、テロ資金、ポンジスキームなど
- **合法（licit）**: 42,019 ノード（約21%） — 取引所、ウォレットプロバイダ、マイニングプールなど
- **不明（unknown）**: 157,205 ノード（約77%）

このデータセットは、**クラス不均衡（class imbalance）** と **大量のラベルなしデータ** という、実世界の不正検知における典型的な課題を含んでいます。

> **注意**: 本ノートブックでは教育目的のため、Ellipticデータセットの特性を模倣した小規模な合成データを使用します。

In [ ]:
# ==================================================
# 合成データの生成
# Ellipticデータセットの特性を模倣した小規模データ
# ==================================================

# パラメータ設定
num_nodes = 500       # ノード数（トランザクション数）
num_edges = 2000      # エッジ数（支払いフロー数）
num_features = 16     # 特徴量の次元数
fraud_ratio = 0.10    # 不正ノードの割合（10%）

# ラベルの生成（不正=1、合法=0）
num_fraud = int(num_nodes * fraud_ratio)
num_legit = num_nodes - num_fraud
labels = torch.zeros(num_nodes, dtype=torch.long)
labels[:num_fraud] = 1  # 先頭num_fraud個を不正とする

# ノードのシャッフル（ランダムに並び替え）
perm = torch.randperm(num_nodes)
labels = labels[perm]

# 特徴量の生成
# 不正ノードと合法ノードで異なる分布からサンプリング
features = torch.zeros(num_nodes, num_features)

for i in range(num_nodes):
    if labels[i] == 1:
        # 不正ノード: 平均がやや高い分布（異常な取引パターンを模倣）
        features[i] = torch.randn(num_features) * 1.5 + 0.5
    else:
        # 合法ノード: 標準的な分布
        features[i] = torch.randn(num_features)

# エッジの生成
# 不正ノード同士はやや接続しやすくする（ホモフィリーの模倣）
fraud_indices = (labels == 1).nonzero(as_tuple=True)[0].numpy()
legit_indices = (labels == 0).nonzero(as_tuple=True)[0].numpy()

edge_list = []
for _ in range(num_edges):
    r = np.random.random()
    if r < 0.15:
        # 不正-不正のエッジ（ホモフィリー）
        src = np.random.choice(fraud_indices)
        dst = np.random.choice(fraud_indices)
    elif r < 0.30:
        # 不正-合法のエッジ
        src = np.random.choice(fraud_indices)
        dst = np.random.choice(legit_indices)
    else:
        # 合法-合法のエッジ
        src = np.random.choice(legit_indices)
        dst = np.random.choice(legit_indices)
    # 自己ループを避ける
    while src == dst:
        dst = np.random.choice(range(num_nodes))
    edge_list.append([src, dst])

# 無向グラフとして両方向のエッジを追加
edge_index_np = np.array(edge_list).T
edge_index = torch.tensor(
    np.concatenate([edge_index_np, edge_index_np[[1, 0]]], axis=1),
    dtype=torch.long
)

# PyTorch GeometricのDataオブジェクトとして構築
data = Data(
    x=features,
    edge_index=edge_index,
    y=labels,
)

print("=" * 60)
print("合成データの概要")
print("=" * 60)
print(f"ノード数: {data.num_nodes}")
print(f"エッジ数: {data.num_edges} (無向グラフとしての実質エッジ数: {data.num_edges // 2})")
print(f"特徴量の次元数: {data.num_node_features}")
print(f"不正ノード数: {(labels == 1).sum().item()} ({(labels == 1).sum().item() / num_nodes * 100:.1f}%)")
print(f"合法ノード数: {(labels == 0).sum().item()} ({(labels == 0).sum().item() / num_nodes * 100:.1f}%)")
print(f"データオブジェクト: {data}")

In [ ]:
# ==================================================
# データ探索: 基本統計量とクラス分布の可視化
# ==================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- クラス分布の棒グラフ ---
class_counts = [
    (labels == 0).sum().item(),
    (labels == 1).sum().item(),
]
class_names = ['合法 (Licit)', '不正 (Illicit)']
colors = ['#2196F3', '#F44336']

bars = axes[0].bar(class_names, class_counts, color=colors, edgecolor='black', linewidth=0.8)
for bar, count in zip(bars, class_counts):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 5,
        f'{count}
({count / num_nodes * 100:.1f}%)',
        ha='center', va='bottom', fontsize=12, fontweight='bold'
    )
axes[0].set_title('クラス分布', fontsize=14, fontweight='bold')
axes[0].set_ylabel('ノード数', fontsize=12)
axes[0].set_ylim(0, max(class_counts) * 1.2)

# --- 特徴量の平均値比較 ---
fraud_mask = labels == 1
legit_mask = labels == 0
fraud_feat_mean = features[fraud_mask].mean(dim=0).numpy()
legit_feat_mean = features[legit_mask].mean(dim=0).numpy()

x_pos = np.arange(num_features)
width = 0.35
axes[1].bar(x_pos - width/2, legit_feat_mean, width, label='合法', color='#2196F3', alpha=0.8)
axes[1].bar(x_pos + width/2, fraud_feat_mean, width, label='不正', color='#F44336', alpha=0.8)
axes[1].set_title('特徴量の平均値比較（クラス別）', fontsize=14, fontweight='bold')
axes[1].set_xlabel('特徴量ID', fontsize=12)
axes[1].set_ylabel('平均値', fontsize=12)
axes[1].set_xticks(x_pos)
axes[1].legend(fontsize=11)
axes[1].axhline(y=0, color='gray', linestyle='--', linewidth=0.5)

plt.suptitle('合成データの探索的分析', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# グラフの基本統計量
G_nx = nx.Graph()
G_nx.add_nodes_from(range(num_nodes))
edge_list_for_nx = edge_index.t().numpy()
for e in edge_list_for_nx:
    if e[0] < e[1]:  # 重複を避ける
        G_nx.add_edge(e[0], e[1])

degrees = [d for _, d in G_nx.degree()]
print("
" + "=" * 60)
print("グラフの基本統計量")
print("=" * 60)
print(f"平均次数: {np.mean(degrees):.2f}")
print(f"最大次数: {np.max(degrees)}")
print(f"最小次数: {np.min(degrees)}")
print(f"孤立ノード数: {sum(1 for d in degrees if d == 0)}")
print(f"連結成分数: {nx.number_connected_components(G_nx)}")

## GCNモデルの定義

**Graph Convolutional Network (GCN)** は、Kipf & Welling (2017) によって提案されたグラフニューラルネットワークの基本的なアーキテクチャです。
各層での更新ルールは以下の通りです:

15555H^{(l+1)} = \sigma\left(\tilde{D}^{-1/2} \tilde{A} \tilde{D}^{-1/2} H^{(l)} W^{(l)}\right)15555

ここで:
- $\tilde{A} = A + I_N$ は自己ループ付きの隣接行列
- $\tilde{D}$ は $\tilde{A}$ の次数行列
- ^{(l)}$ は第 $ 層のノード特徴行列
- ^{(l)}$ は学習可能な重み行列
- $\sigma$ は活性化関数（ReLUなど）

直感的には、各ノードが近傍ノードの特徴量を収集・集約し、自分自身の表現を更新するメッセージパッシングの仕組みです。

以下では、**2層のGCN** を定義します。

In [ ]:
class GCNFraudDetector(torch.nn.Module):
    """
    2層のGraph Convolutional Networkによる不正検知モデル

    アーキテクチャ:
        入力層 (num_features) -> GCNConv -> ReLU -> Dropout
        -> 隠れ層 (hidden_dim) -> GCNConv -> 出力層 (2クラス)
    """

    def __init__(self, num_features, hidden_dim=32, dropout=0.5):
        super(GCNFraudDetector, self).__init__()
        # 第1層: 入力特徴量 -> 隠れ層
        self.conv1 = GCNConv(num_features, hidden_dim)
        # 第2層: 隠れ層 -> 出力（2クラス分類）
        self.conv2 = GCNConv(hidden_dim, 2)
        # ドロップアウト率
        self.dropout = dropout

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        # 第1層: グラフ畳み込み + ReLU + Dropout
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        # 第2層: グラフ畳み込み（活性化関数なし）
        x = self.conv2(x, edge_index)

        # 対数ソフトマックスで確率分布を出力
        return F.log_softmax(x, dim=1)

# モデルのインスタンス化
model = GCNFraudDetector(
    num_features=num_features,
    hidden_dim=32,
    dropout=0.5
).to(device)

print("モデル構造:")
print(model)
print(f"
パラメータ総数: {sum(p.numel() for p in model.parameters()):,}")
print(f"学習可能パラメータ数: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# ==================================================
# データの分割: 訓練 / 検証 / テスト
# ==================================================

# 全ノードのインデックス
indices = np.arange(num_nodes)

# まず訓練+検証 (80%) と テスト (20%) に分割
train_val_idx, test_idx = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=labels.numpy()
)

# 次に訓練 (60%) と 検証 (20%) に分割
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=0.25, random_state=42,
    stratify=labels.numpy()[train_val_idx]
)

# マスクの作成（PyTorch Geometricの標準的な方法）
train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)

train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True

# データオブジェクトにマスクを追加
data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

print("=" * 60)
print("データ分割の結果")
print("=" * 60)
for name, mask in [('訓練', train_mask), ('検証', val_mask), ('テスト', test_mask)]:
    n_total = mask.sum().item()
    n_fraud = (labels[mask] == 1).sum().item()
    n_legit = (labels[mask] == 0).sum().item()
    print(f"  {name}: {n_total}ノード "
          f"(合法: {n_legit}, 不正: {n_fraud}, "
          f"不正率: {n_fraud / n_total * 100:.1f}%)")

In [ ]:
# ==================================================
# 訓練ループ
# ==================================================

# データをデバイスに転送
data = data.to(device)

# オプティマイザの設定
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

# クラス不均衡に対応するための重み付き損失関数
# 不正クラスの重みを大きくする
class_weights = torch.tensor(
    [1.0, num_legit / num_fraud],  # 合法:1.0, 不正:合法数/不正数
    dtype=torch.float
).to(device)
print(f"クラス重み: 合法={class_weights[0]:.1f}, 不正={class_weights[1]:.1f}")

# 訓練履歴の記録用
train_losses = []
val_losses = []
train_accs = []
val_accs = []

num_epochs = 100

print(f"
訓練開始（{num_epochs}エポック）")
print("-" * 70)

for epoch in range(1, num_epochs + 1):
    # --- 訓練フェーズ ---
    model.train()
    optimizer.zero_grad()
    out = model(data)

    # 訓練ノードのみで損失を計算
    train_loss = F.nll_loss(
        out[data.train_mask], data.y[data.train_mask], weight=class_weights
    )
    train_loss.backward()
    optimizer.step()

    # 訓練精度の計算
    pred_train = out[data.train_mask].argmax(dim=1)
    train_acc = (pred_train == data.y[data.train_mask]).sum().item() / data.train_mask.sum().item()

    # --- 検証フェーズ ---
    model.eval()
    with torch.no_grad():
        out = model(data)
        val_loss = F.nll_loss(
            out[data.val_mask], data.y[data.val_mask], weight=class_weights
        )
        pred_val = out[data.val_mask].argmax(dim=1)
        val_acc = (pred_val == data.y[data.val_mask]).sum().item() / data.val_mask.sum().item()

    # 履歴の記録
    train_losses.append(train_loss.item())
    val_losses.append(val_loss.item())
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    # 10エポックごとにログを出力
    if epoch % 10 == 0 or epoch == 1:
        print(
            f"  エポック {epoch:3d}/{num_epochs}: "
            f"訓練損失={train_loss.item():.4f}, 訓練精度={train_acc:.4f}, "
            f"検証損失={val_loss.item():.4f}, 検証精度={val_acc:.4f}"
        )

print("-" * 70)
print("訓練完了")

In [ ]:
# ==================================================
# 学習曲線の可視化
# ==================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 損失の推移 ---
epochs_range = range(1, num_epochs + 1)
axes[0].plot(epochs_range, train_losses, label='訓練損失', color='#2196F3', linewidth=2)
axes[0].plot(epochs_range, val_losses, label='検証損失', color='#F44336', linewidth=2)
axes[0].set_xlabel('エポック', fontsize=12)
axes[0].set_ylabel('損失', fontsize=12)
axes[0].set_title('損失の推移', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# --- 精度の推移 ---
axes[1].plot(epochs_range, train_accs, label='訓練精度', color='#2196F3', linewidth=2)
axes[1].plot(epochs_range, val_accs, label='検証精度', color='#F44336', linewidth=2)
axes[1].set_xlabel('エポック', fontsize=12)
axes[1].set_ylabel('精度', fontsize=12)
axes[1].set_title('精度の推移', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1.05)

plt.suptitle('GCNモデルの学習曲線', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ==================================================
# テストデータでの評価
# ==================================================

model.eval()
with torch.no_grad():
    out = model(data)
    pred = out[data.test_mask].argmax(dim=1).cpu().numpy()
    true = data.y[data.test_mask].cpu().numpy()

# 各種評価指標の計算
acc = accuracy_score(true, pred)
prec = precision_score(true, pred, zero_division=0)
rec = recall_score(true, pred, zero_division=0)
f1 = f1_score(true, pred, zero_division=0)

print("=" * 60)
print("テストデータでの評価結果")
print("=" * 60)
print(f"  正解率 (Accuracy):  {acc:.4f}")
print(f"  適合率 (Precision): {prec:.4f}")
print(f"  再現率 (Recall):    {rec:.4f}")
print(f"  F1スコア:            {f1:.4f}")

print("
--- 詳細な分類レポート ---")
target_names = ['合法 (0)', '不正 (1)']
print(classification_report(true, pred, target_names=target_names, zero_division=0))

# --- 混同行列の可視化 ---
cm = confusion_matrix(true, pred)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
ax.set_title('混同行列（テストデータ）', fontsize=14, fontweight='bold')

# カラーバー
cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('サンプル数', fontsize=11)

# 軸ラベル
tick_marks = [0, 1]
ax.set_xticks(tick_marks)
ax.set_yticks(tick_marks)
ax.set_xticklabels(target_names, fontsize=11)
ax.set_yticklabels(target_names, fontsize=11)
ax.set_xlabel('予測ラベル', fontsize=12)
ax.set_ylabel('真のラベル', fontsize=12)

# セル内に値を表示
thresh = cm.max() / 2.0
for i in range(2):
    for j in range(2):
        ax.text(
            j, i, format(cm[i, j], 'd'),
            ha='center', va='center',
            color='white' if cm[i, j] > thresh else 'black',
            fontsize=16, fontweight='bold'
        )

plt.tight_layout()
plt.show()

# 不正検知における評価指標の解釈
print("
--- 不正検知における評価指標の解釈 ---")
print(f"  適合率 (Precision) = {prec:.4f}: "
      f"不正と予測したもののうち、実際に不正だった割合")
print(f"  再現率 (Recall)    = {rec:.4f}: "
      f"実際の不正のうち、正しく検出できた割合")
print(f"  F1スコア           = {f1:.4f}: "
      f"適合率と再現率の調和平均")

## GAT (Graph Attention Network) の概念

GCNでは全ての近傍ノードからの情報を **均等に** 集約しますが、実際の不正検知では一部の接続がより重要な場合があります。

**Graph Attention Network (GAT)** （Velickovic et al., 2018）は、アテンション機構を導入し、各エッジに **異なる重要度（アテンション係数）** を学習します。

### アテンションの計算

15555\alpha_{ij} = \frac{\exp\left(\text{LeakyReLU}\left(\mathbf{a}^T [\mathbf{W}\mathbf{h}_i \| \mathbf{W}\mathbf{h}_j]\right)\right)}{\sum_{k \in \mathcal{N}(i)} \exp\left(\text{LeakyReLU}\left(\mathbf{a}^T [\mathbf{W}\mathbf{h}_i \| \mathbf{W}\mathbf{h}_k]\right)\right)}15555

ここで:
- $\alpha_{ij}$: ノード $ からノード $ へのアテンション係数
- $\mathbf{W}$: 共有の線形変換行列
- $\mathbf{a}$: アテンションベクトル
- $\|$: ベクトルの結合
- $\mathcal{N}(i)$: ノード $ の近傍集合

### 不正検知におけるGATの利点

- **選択的な情報集約**: 不正な取引パターンを持つ近傍ノードにより高い注目度を割り当てる
- **解釈性の向上**: アテンション重みを可視化することで、どの接続が予測に寄与しているかを理解可能
- **マルチヘッドアテンション**: 複数のアテンションヘッドにより、異なる観点から近傍の重要性を評価

以下ではGATモデルを実装し、GCNとの比較を行います。

In [ ]:
class GATFraudDetector(torch.nn.Module):
    """
    Graph Attention Networkによる不正検知モデル

    アテンション機構により、各近傍ノードからの情報に
    異なる重要度を割り当てて集約する。
    """

    def __init__(self, num_features, hidden_dim=32, heads=4, dropout=0.5):
        super(GATFraudDetector, self).__init__()
        # 第1層: マルチヘッドアテンション
        self.conv1 = GATConv(
            num_features, hidden_dim, heads=heads, dropout=dropout
        )
        # 第2層: 単一ヘッドアテンション（出力層）
        self.conv2 = GATConv(
            hidden_dim * heads, 2, heads=1, concat=False, dropout=dropout
        )
        self.dropout = dropout

    def forward(self, data, return_attention=False):
        x, edge_index = data.x, data.edge_index

        # 第1層: GAT + ELU + Dropout
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        # 第2層: GAT（アテンション重みも返すオプション）
        if return_attention:
            x, (edge_index_out, attention_weights) = self.conv2(
                x, edge_index, return_attention_weights=True
            )
            return F.log_softmax(x, dim=1), attention_weights
        else:
            x = self.conv2(x, edge_index)
            return F.log_softmax(x, dim=1)


# GATモデルのインスタンス化と訓練
gat_model = GATFraudDetector(
    num_features=num_features,
    hidden_dim=8,
    heads=4,
    dropout=0.5
).to(device)

print("モデル構造 (GAT):")
print(gat_model)
print(f"
パラメータ総数: {sum(p.numel() for p in gat_model.parameters()):,}")

# GATの訓練
gat_optimizer = torch.optim.Adam(gat_model.parameters(), lr=0.005, weight_decay=5e-4)

gat_train_losses = []
gat_val_losses = []

print(f"
GATモデルの訓練開始（{num_epochs}エポック）")
print("-" * 70)

for epoch in range(1, num_epochs + 1):
    gat_model.train()
    gat_optimizer.zero_grad()
    out = gat_model(data)
    loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask], weight=class_weights)
    loss.backward()
    gat_optimizer.step()
    gat_train_losses.append(loss.item())

    gat_model.eval()
    with torch.no_grad():
        out = gat_model(data)
        val_loss = F.nll_loss(out[data.val_mask], data.y[data.val_mask], weight=class_weights)
        gat_val_losses.append(val_loss.item())

    if epoch % 20 == 0 or epoch == 1:
        pred_val = out[data.val_mask].argmax(dim=1)
        val_acc = (pred_val == data.y[data.val_mask]).sum().item() / data.val_mask.sum().item()
        print(f"  エポック {epoch:3d}: 訓練損失={loss.item():.4f}, 検証損失={val_loss.item():.4f}, 検証精度={val_acc:.4f}")

print("-" * 70)

# GATのテスト評価
gat_model.eval()
with torch.no_grad():
    out = gat_model(data)
    gat_pred = out[data.test_mask].argmax(dim=1).cpu().numpy()

gat_acc = accuracy_score(true, gat_pred)
gat_prec = precision_score(true, gat_pred, zero_division=0)
gat_rec = recall_score(true, gat_pred, zero_division=0)
gat_f1 = f1_score(true, gat_pred, zero_division=0)

print("
" + "=" * 60)
print("モデル比較: GCN vs GAT")
print("=" * 60)
comparison_df = pd.DataFrame({
    '指標': ['正解率', '適合率', '再現率', 'F1スコア'],
    'GCN': [acc, prec, rec, f1],
    'GAT': [gat_acc, gat_prec, gat_rec, gat_f1],
})
print(comparison_df.to_string(index=False, float_format='{:.4f}'.format))

# アテンション重みの可視化
print("
--- アテンション重みの分析 ---")
print("アテンション重みは、各エッジにおいて近傍ノードがどの程度重要かを示します。")
print("不正ノードの検出において、どの接続が予測に寄与しているかを")
print("解釈するための手がかりとなります。")

## 考察

### GNNが不正検知に有効な理由

不正検知においてGNNが従来手法よりも優れている主な理由は以下の通りです:

1. **ホモフィリー（同類選好性）の活用**: 不正行為者は他の不正行為者と取引する傾向があり、GNNはこのグラフ上のパターンを自然に学習可能
2. **メッセージパッシングによる特徴拡張**: 近傍ノードの情報を集約することで、個々のノードの特徴量だけでは捕捉できない不正パターンを検出
3. **ラベル伝搬**: ラベルが付いていないノードも、グラフ構造を通じてラベル付きノードの情報を間接的に利用可能
4. **多ホップ推論**: GNNの層を重ねることで、より遠くのノードからの情報も集約でき、複雑な不正ネットワークを検出

### クラス不均衡への対処法

不正検知では、不正データは全体のごく一部（典型的には1〜5%）であり、クラス不均衡が大きな課題となります。以下の対処法が研究されています:

| 手法 | 説明 |
|---|---|
| **重み付き損失関数** | 少数クラスの損失に大きな重みを付与（本ノートブックで使用） |
| **オーバーサンプリング** | SMOTEなどにより少数クラスのサンプルを増強 |
| **GraphSMOTE** | グラフ構造を考慮した合成ノードの生成 |
| **Focal Loss** | 容易なサンプルの損失を下げ、困難なサンプルに集中 |
| **アンダーサンプリング** | 多数クラスのサンプルを削減 |
| **異常検知アプローチ** | 不正検知を二値分類ではなく異常検知として定式化 |

### 実世界での課題

実際の金融不正検知システムではGNNを適用する際に、以下の課題を考慮する必要があります:

1. **スケーラビリティ**: Ellipticデータセットでも約20万ノードですが、実際の金融ネットワークは数億規模になり得ます。ミニバッチ学習（GraphSAGEなど）やサブグラフサンプリングが必要です。

2. **時系列性**: 金融取引は時系列データであり、不正の手口は時間とともに変化します。Temporal GNN（例: TGN, TGAT）などの時系列対応モデルが研究されています。

3. **敵対的攻撃**: 不正行為者は意図的に検知を回避する行動を取るため、モデルのロバスト性が重要です。グラフ構造への敵対的操作（エッジの追加・削除）に対する防御機構が必要です。

4. **解釈性と説明可能性**: 金融規制の観点から、「なぜ不正と判定したのか」を説明できることが求められます。GATのアテンション重みやGNNExplainerなどの説明手法が活用されます。

5. **ラベルの希少性とノイズ**: 実世界では、不正ラベルの取得には調査・法執行機関の確認が必要であり、ラベルにノイズが含まれることもあります。半教師あり学習や自己教師あり学習のアプローチが有効です。

6. **プライバシーと倫理**: 取引データは機密性が高く、個人情報保護規制（GDPR、個人情報保護法など）への準拠が必要です。連合学習（Federated Learning）とGNNの組み合わせも研究されています。

### まとめ

本ノートブックでは、GCNとGATを用いたグラフベースの不正検知の基本的なワークフローを実装しました。GNNは取引ネットワークのグラフ構造を直接学習に活用できる点で、従来の機械学習手法と比較して大きな利点があります。しかし、実運用にはスケーラビリティ、時系列対応、解釈性などの課題が残っており、活発な研究が続いています。

詳細は以下のドキュメントを参照してください:
- [03-computer-science.md](../03-computer-science.md) — GNNの理論的背景
- [05-emerging-fields.md](../05-emerging-fields.md) — 不正検知・AML分野の最新動向